# Train all models on your own GPU (CUDA)

Put this notebook **inside the `traffic_project` folder** (next to `config.py`) and open it with Jupyter / VS Code using your `dl` virtual environment.

Everything runs in this notebook: data check, training of the 4 deep models on METR-LA, evaluation, and figures. Results go to `results/`, figures to `figures/`, trained weights to `checkpoints/` (the web app reads them from there).

In [ ]:
# 1. Check the GPU
import sys, torch
print('python :', sys.executable)
print('torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(no GPU found, will use CPU)')

In [ ]:
# 2. Install the other packages into THIS environment (skip if already installed)
%pip install -q numpy pandas matplotlib plotly streamlit

In [ ]:
# 3. Training settings. On a GPU we can afford full epochs.
EPOCHS = 50          # maximum; early stopping usually ends sooner
BATCHES = None       # None = use every training batch in each epoch
DATASETS = ['metr-la']   # our project uses METR-LA only
MODELS = ['lstm', 'stgcn', 'static_st', 'adaptive_st']

In [ ]:
# 4. Module 1: data summary + exploration figures
from m1_data.dataset import prepare_dataset
%run -m m1_data.explore

In [ ]:
# 5. Module 2 + 3: train every model on every dataset
from m3_training.train import train
summaries = {}
for ds in DATASETS:
    data = prepare_dataset(ds)
    for m in MODELS:
        summaries[(ds, m)] = train(ds, m, epochs=EPOCHS, max_batches=BATCHES, data=data)

In [ ]:
# 6. Module 3: test-set evaluation (MAE / RMSE / MAPE at 15, 30, 60 min)
import pandas as pd
from m3_training.evaluate import evaluate_all
for ds in DATASETS:
    print('\n', ds)
    evaluate_all(ds)
    display(pd.read_csv(f'results/{ds}_results.csv', index_col=0))

In [ ]:
# 7. Module 3: figures for the slides
for ds in DATASETS:
    %run -m m3_training.plots --dataset {ds}

In [ ]:
# 8. Show the main result figures
from IPython.display import Image, display
for f in ['m3_horizon_mae_metr-la.png', 'm3_forecast_example_metr-la.png', 'm2_graphs_metr-la.png']:
    display(Image(f'figures/{f}'))

## Web app
In a terminal, from the `traffic_project` folder:
```bash
streamlit run webapp/app.py
```